## Data Orchestration & Workflow Orchestration

**Data orchestration and workflow orchestration** are related but distinct concepts:
  *  **Data orchestration** focuses on **coordinating, automating and managing the flow of data** across systems, ensuring its movement, transformation, and integration. 
     *  **Key activities:** ETL (Extract, Transform, Load) processes, data synchronization, and ensuring data availability where and when needed. 
     *  **Example:** Managing the flow of data from a source database to a data warehouse, including cleaning, transforming, and loading it. 
  * **Workflow orchestration**, on the other hand, manages the sequence of tasks in a process, ensuring they are executed in the correct order. (**broader execution of tasks**)
    * **Key activities:** Task scheduling, dependency management, and monitoring task status. 
    * **Example:** Automating the steps in a software development pipeline, 



#### **Concept and benefits of workflow orchestration in data engineering and machine learning pipelines:**
* **The Problem:** Data pipelines often involve multiple steps (ingestion, cleaning, transformation, model training, prediction, reporting). These steps have dependencies and need to be executed in a specific order. Manual execution is prone to errors, time-consuming, and difficult to scale.
* **The Solution --> Workflow Orchestration:** Think of it as a conductor for an orchestra. It ensures each instrument (task) plays at the right time, in the right order, and if something goes wrong, it helps identify the issue and allows for recovery.
* **Benefits:**
    * **Automation:** Eliminates manual execution, reducing human error and freeing up time.
    * **Reliability:** Ensures tasks run consistently, even in case of failures (retries).
    * **Scalability:** Manages increasingly complex and numerous workflows.
    * **Visibility:** Provides a clear overview of pipeline status, progress, and logs.
    * **Reproducibility:** Workflows defined as code can be version-controlled, ensuring consistent results.
    * **Collaboration:** Teams can share and manage workflows more effectively.


## AirFlow: open-source workflow management platform (de facto Orchestrator)


#### **Introduction to Apache Airflow**
* **Developed by Airbnb in 2015**, now a thriving Apache Software Foundation project.
* **"Programmatically author, schedule and monitor workflows."** This is Airflow's core promise.
  * **The ability to write pipelines-as-code with Python**, making it accessible for data scientists and engineers.
  * It is a **community driven** and **open-source**
  * **Built-in observability features** in Airflow (alerts, retry mechanisms, etc.) 
  * **Scheduling** to schedule your workflows, based on time and events (data landing, etc.)
  * **Extensibility to connect to external systems** with integrations, called packages, Python packages that extend the core functionality of Airflow to customize your instances. Airflow packages are modular components that allow you to tailor your Airflow environment to your specific needs by installing only the necessary integrations. Since it is open-source you can do it on your own.

* **Considerations:**
  * It's not a data processing engine itself (like Spark). Instead, it **tells *other* engines what to do and when**.
  * **Not good for Streaming**,  you can use it with Kafka which may store data somewhere and use Airflow (it mainly for batch processing) 
  * By default is **scalable**, however the are always limitations based on the underlining infrastructure



#### **Key Airflow concepts:**
![image.png](../artifacts/01_airflow_dag_concepts.png)
  * **DAGs (Directed Acyclic Graphs):** The heart of Airflow. A DAG is a collection of all the tasks you want to run, organized in a way that shows their relationships and dependencies. AKA a Dag is a data pipeline.
      * "Directed": Tasks flow in one direction, no loops. there is a direction/order of operations that definde dependencies. 
      * "Acyclic": No task can depend on itself or create a loop.
      * "Graph": A visual representation of tasks and their connections.
      * Defined in Python code.
  * **Tasks:**
      * Individual units of work within a DAG (a node in the graph). 
      * Each task performs a specific action (e.g., download data, run a SQL query, execute a Spark job, train a model).
      * Tasks are instances of operators.
  * **Operators:**
      * Pre-built templates that define how a task should perform a specific action. Airflow provides many operators for common operations (BashOperator, PythonOperator, EmailOperator, and crucially for us, operators for GCP services).
      * Example: `BashOperator` to run a shell command, `PythonOperator` to execute a Python function.
      * There are three main types of operators:
        * Action: Perform a specific action such as running code or a bash command
        * Transfer: Perform transfer operations that move data between two systems 
        * Sensor: Wait for a specific condition to be met (e.g., waiting for a file to be present) before running the next task


#### Airflow Architecture


* **How Airflow enables scheduling, monitoring, and retrying tasks:**
    * **Scheduler:** The Airflow component that monitors DAGs and triggers tasks based on their dependencies and schedules.
    * **Webserver:** Provides a user interface to visualize DAGs, monitor task status, view logs, and manually trigger DAGs.
    * **Workers:** The processes that actually execute the tasks defined in your DAGs.
    * **Retries:** Airflow can be configured to automatically retry failed tasks a specified number of times, improving pipeline resilience.
  
* Airflow has seven main components:
  * The **API server** for serving the Airflow UI and handling task execution, providing endpoints for Workers to communicate with the Airflow DB.
  * The **metadata database** is used to store all metadata (e.g., users that have access to instances, tasks) related to your Airflow instance, the only other components that communicate with it are the Scheduler and the API Server.
  * The **scheduler** for monitoring and scheduling your pipelines, when dependencies are fulfilled (ex. time, events).
  * The **executor** defines *how and on which system* tasks are executed. 
  * The **DAG File Processor** that retrieves and parses DAG files.
  * The **queue** for holding tasks that are ready to be executed, defines the **execution task order**.
  * The **worker(s)** for executing instructions defined in a task, definde by the executor.
  * **Trigger**: process running asyncio to support deferrable operations

  ![image.png](../artifacts/01_airflow_architecture.png)


* **Airflow steps:**
  1. User add a new Dag in the Dag folder 
  2. The DAG File Processor constantly scans the DAGs directory for new files. The default time is every 5 minutes.
  3. After the DAG File Processor detects a new DAG, the DAG is processed and serialized into the metadata database.
  4. The scheduler checks for DAGs that are ready to run in the metadata database. The default time is every 5 seconds. If it finds one it creates a schedule instance/task
  5. Once a DAG is ready to run, its tasks are put into the executor's queue.
  6. Once a worker is available, it will retrieve a task to execute from the queue.
  7. The worker will then execute the task and constantly updates the status of the task 
![alt text](<images/workflow in airflow.PNG>)



### Methods for Defining Dependencies


Setting up dependencies in Apache Airflow is fundamental to defining the order in which your tasks run within a Directed Acyclic Graph (DAG). Airflow uses the concept of "upstream" and "downstream" tasks, where a downstream task can only run once all its upstream tasks have completed successfully.

There are several ways to define these dependencies, from simple to more complex scenarios.

#### Core Concepts

  * **Task:** The basic unit of work in Airflow, created by instantiating an Operator (e.g., `BashOperator`, `PythonOperator`).
  * **DAG (Directed Acyclic Graph):** A collection of tasks organized in a way that shows their relationships and dependencies. The "acyclic" part means there are no loops; tasks always progress forward.
  * **Upstream/Downstream:** If Task A must complete before Task B can start, Task A is upstream of Task B, and Task B is downstream of Task A.
  
#### Primary ways to set up dependencies in Airflow:

**1. Using Bitshift Operators (`>>` and `<<`) - Most Common and Recommended**

This is the most Pythonic and readable way to define dependencies.

  * **`>>` (downstream operator):** Sets the task on the left as upstream of the task on the right.
  * **`<<` (upstream operator):** Sets the task on the left as downstream of the task on the right.

**Example:**


In [ ]:
# Example Airflow / GCP code snippet
from airflow.models.dag import DAG
from airflow.operators.bash import BashOperator
from datetime import datetime

with DAG(
    dag_id='dependency_example_bitshift',
    start_date=datetime(2023, 1, 1),
    schedule_interval=None,
    catchup=False,
    tags=['example'],
) as dag:
    task1 = BashOperator(
        task_id='task_1',
        bash_command='echo "Running Task 1"',
    )

    task2 = BashOperator(
        task_id='task_2',
        bash_command='echo "Running Task 2"',
    )

    task3 = BashOperator(
        task_id='task_3',
        bash_command='echo "Running Task 3"',
    )

    task4 = BashOperator(
        task_id='task_4',
        bash_command='echo "Running Task 4"',
    )

    # --- Defining dependencies using bitshift operators ---

    # task1 runs before task2
    task1 >> task2

    # task2 runs before task3
    task2 >> task3

    # task4 runs after task1
    task4 << task1

    # Multiple dependencies: task3 and task4 both run before task5
    # (assuming you had a task5)
    # [task3, task4] >> task5


**Complex Chains with Bitshift:**

You can chain multiple tasks:


In [ ]:
# Example Airflow / GCP code snippet
task1 >> task2 >> task3
# This is equivalent to:
# task1 >> task2
# task2 >> task3


In [ ]:
# Multiple tasks feeding into one:
[task1, task2] >> task3
# This means both task1 AND task2 must complete before task3 starts.
# Equivalent to:
# task1 >> task3
# task2 >> task3

# One task feeding into multiple:
task1 >> [task2, task3]
# This means task1 completes, then task2 and task3 can run in parallel.
# Equivalent to:
# task1 >> task2
# task1 >> task3


**2. Using `set_upstream()` and `set_downstream()` Methods**

These are explicit methods on the Task objects. They achieve the same result as the bitshift operators but are generally less concise for complex DAGs.

  * `task.set_upstream(other_task)`: Makes `other_task` run before `task`.
  * `task.set_downstream(other_task)`: Makes `task` run before `other_task`.

**When to use `set_upstream()`/`set_downstream()`:**

  * When generating dependencies programmatically (e.g., in a loop where the upstream/downstream task names are variables).
  * For very simple, explicit relationships where readability is prioritized for individual task definitions.
  * When integrating with external systems that might provide dependency information in a different format.
  
**Example:**


In [ ]:
# Example Airflow / GCP code snippet
from airflow.models.dag import DAG
from airflow.operators.bash import BashOperator
from datetime import datetime

with DAG(
    dag_id='dependency_example_set_methods',
    start_date=datetime(2023, 1, 1),
    schedule_interval=None,
    catchup=False,
    tags=['example'],
) as dag:
    task_a = BashOperator(
        task_id='task_a',
        bash_command='echo "Running Task A"',
    )

    task_b = BashOperator(
        task_id='task_b',
        bash_command='echo "Running Task B"',
    )

    task_c = BashOperator(
        task_id='task_c',
        bash_command='echo "Running Task C"',
    )

    # --- Defining dependencies using set_upstream/set_downstream ---

    # task_b runs after task_a
    task_b.set_upstream(task_a)

    # task_c runs after task_b
    task_b.set_downstream(task_c) # Same as task_c.set_upstream(task_b)


**3. Using `cross_downstream()` (Less Common for Direct Dependencies)**

This method is less about direct parent-child dependencies and more about complex "sub-DAG" like structures or when you need to define dependencies across lists of tasks. It's often used with `chain()` as well.

It's essentially a syntactic sugar for applying a list of tasks as upstream to another list of tasks.


In [ ]:
# Example Airflow / GCP code snippet
from airflow.models.dag import DAG
from airflow.operators.bash import BashOperator
from datetime import datetime

with DAG(
    dag_id='dependency_example_cross_downstream',
    start_date=datetime(2023, 1, 1),
    schedule_interval=None,
    catchup=False,
    tags=['example'],
) as dag:
    start = BashOperator(task_id='start', bash_command='echo "Start"')
    group1_tasks = [
        BashOperator(task_id='g1_task_1', bash_command='echo "G1 Task 1"'),
        BashOperator(task_id='g1_task_2', bash_command='echo "G1 Task 2"')
    ]
    group2_tasks = [
        BashOperator(task_id='g2_task_1', bash_command='echo "G2 Task 1"'),
        BashOperator(task_id='g2_task_2', bash_command='echo "G2 Task 2"')
    ]
    end = BashOperator(task_id='end', bash_command='echo "End"')

    # All tasks in group1_tasks must complete before any task in group2_tasks starts
    start >> group1_tasks
    group1_tasks >> group2_tasks # This is the cross_downstream in action behind the scenes
    group2_tasks >> end

    # More explicitly, this is how cross_downstream works:
    # Task 1 from G1 -> Task 1 from G2
    # Task 1 from G1 -> Task 2 from G2
    # Task 2 from G1 -> Task 1 from G2
    # Task 2 from G1 -> Task 2 from G2


**4. Using `chain()` and `cross_downstream()` from `airflow.models.baseoperator`**

These helper functions are useful for more complex or programmatic dependency definitions, especially with lists of tasks.

  * **`chain(*tasks)`:** Creates a linear dependency chain.
    `chain(task1, task2, task3)` is equivalent to `task1 >> task2 >> task3`.
  * **`cross_downstream(from_tasks, to_tasks)`:** Sets up dependencies so that *every* task in `from_tasks` is upstream of *every* task in `to_tasks`. This can quickly create a "mesh" of dependencies.

**Example:**


In [ ]:
# Example Airflow / GCP code snippet
from airflow.models.dag import DAG
from airflow.operators.bash import BashOperator
from airflow.models.baseoperator import chain, cross_downstream # Import them
from datetime import datetime

with DAG(
    dag_id='dependency_example_chain_cross',
    start_date=datetime(2023, 1, 1),
    schedule_interval=None,
    catchup=False,
    tags=['example'],
) as dag:
    t_start = BashOperator(task_id='start', bash_command='echo "Start"')
    t_middle1 = BashOperator(task_id='middle_1', bash_command='echo "Middle 1"')
    t_middle2 = BashOperator(task_id='middle_2', bash_command='echo "Middle 2"')
    t_end = BashOperator(task_id='end', bash_command='echo "End"')

    # Chain example
    chain(t_start, t_middle1, t_end) # start >> middle_1 >> end

    # Cross_downstream example (let's add some more tasks for clarity)
    branch1_tasks = [
        BashOperator(task_id='b1_task_a', bash_command='echo "Branch 1A"'),
        BashOperator(task_id='b1_task_b', bash_command='echo "Branch 1B"')
    ]
    branch2_tasks = [
        BashOperator(task_id='b2_task_x', bash_command='echo "Branch 2X"'),
        BashOperator(task_id='b2_task_y', bash_command='echo "Branch 2Y"')
    ]

    # All tasks in branch1_tasks must complete before any task in branch2_tasks starts
    cross_downstream(from_tasks=branch1_tasks, to_tasks=branch2_tasks)
    # This creates 4 dependencies:
    # b1_task_a >> b2_task_x
    # b1_task_a >> b2_task_y
    # b1_task_b >> b2_task_x
    # b1_task_b >> b2_task_y



#### Best Practices

  * **Use Bitshift Operators (`>>`, `<<`) for clarity:** For most DAGs, these provide the most readable and concise way to define dependencies.
  * **Visualize your DAG:** After defining dependencies, use the Airflow UI's Graph View to confirm that your dependencies are set up as intended. This is crucial for debugging.
  * **Modularize with Task Groups (Airflow 2.x):** For complex DAGs, use `TaskGroup` to organize related tasks and their internal dependencies. This makes the DAG definition cleaner and easier to manage, as you can then define dependencies between entire task groups.
  * **Mind `trigger_rule` (Advanced):** By default, a task only runs if all its upstream tasks succeed (`all_success` trigger rule). You can change this behavior (e.g., `one_success`, `all_done`, `none_failed_min_one_success`) for more advanced branching and error handling scenarios. This isn't directly "setting up dependencies" but affects how they are evaluated.


### **Airflow DAG and @dag Evolution**

In Apache Airflow 2.0 and later versions, you may choose between `from airflow import DAG` (traditional syntax) and `@dag` (pythonic syntax). The answer is: **`@dag` decorator is the best solution for new DAGs**.


#### The Traditional Approach (`from airflow import DAG`)

This is the classic way to define a DAG in Apache Airflow, using a context manager (`with DAG(...)`).
**Characteristics:**

  * **Verbose:** Requires more lines of boilerplate code.
  * **Explicit:** The DAG definition is enclosed within the `with` block.
  * **Backward Compatibility:** The only way to define DAGs in Airflow 1.x.


**Example:**


In [ ]:
# Example Airflow / GCP code snippet
from airflow import DAG
from airflow.operators.bash import BashOperator
from datetime import datetime

with DAG(
    dag_id='traditional_dag_example',
    start_date=datetime(2023, 1, 1),
    schedule_interval=None,
    catchup=False,
    tags=['traditional']
) as dag:

    task_1 = BashOperator(
        task_id='print_date',
        bash_command='echo "Date: $(date)"',
    )

    task_2 = BashOperator(
        task_id='print_hello',
        bash_command='echo "Hello from Task 2"',
    )

    task_1 >> task_2 # Sets the dependency


#### The Modern Approach (`@dag` decorator)

This approach, introduced in Airflow 2.0, uses functional programming and a decorator to define a DAG.

**Characteristics:**

  * **Concise:** Removes most of the boilerplate code. The function itself defines the DAG.
  * **More Pythonic:** Better aligns with modern Python coding style.
  * **Seamlessly integrates with TaskFlow API:** This is its main strength, allowing for even more readable task dependency definitions.


**Code Example:**


In [ ]:
# Example Airflow / GCP code snippet
from airflow.decorators import dag
from airflow.operators.bash import BashOperator
from datetime import datetime

@dag(
    start_date=datetime(2023, 1, 1),
    schedule_interval=None,
    catchup=False,
    tags=['modern']
)
def modern_dag_example():

    task_1 = BashOperator(
        task_id='print_date',
        bash_command='echo "Date: $(date)"',
    )

    task_2 = BashOperator(
        task_id='print_hello',
        bash_command='echo "Hello from Task 2"',
        )

    task_1 >> task_2 # Sets the dependency

# Create a DAG instance by calling the function
my_dag = modern_dag_example()


#### The Verdict: `@dag` is Clearly Superior

The main advantage of the `@dag` decorator is its integration with the **TaskFlow API**, which allows you to define tasks using the `@task` decorator.

**TaskFlow API Example:**


In [ ]:
# Example Airflow / GCP code snippet
from airflow.decorators import dag, task
from datetime import datetime

@task
def get_data():
    """Simulates data retrieval."""
    print("Retrieving data...")
    return 42

@task
def process_data(value):
    """Simulates data processing."""
    print(f"Processing value: {value}")
    return value * 2

@dag(start_date=datetime(2023, 1, 1), schedule_interval=None, catchup=False)
def data_pipeline_with_taskflow():

    # Dependencies are explicitly and readably defined
    # The value returned by get_data() is passed to process_data()
    processed_value = process_data(get_data())

data_pipeline_with_taskflow()


As you can see, with the TaskFlow API, dependencies (`get_data` must run before `process_data`) and data passing between tasks become incredibly clear. This is impossible with the traditional approach without more complex operators.
#### Comparative Summary

| Characteristic            | `from airflow import DAG` (Traditional) | `@dag` (Modern)                                  |
| :------------------------ | :-------------------------------------- | :----------------------------------------------- |
| **Syntax** | Class-based, `with` context manager     | Function-based, decorator                        |
| **Conciseness** | ❌ Very verbose                         | ✅ Very concise                                  |
| **Readability** | Good, but requires boilerplate          | ✅ Excellent, especially with `@task`            |
| **Airflow Version** | Airflow 1.x and 2.x                     | Airflow 2.0+ (recommended)                       |
| **TaskFlow Integration** | ❌ Requires manual approach             | ✅ Complete and native                           |
| **Recommended Use** | Maintaining legacy DAGs.                | **All new DAGs in Airflow 2.0+**.                |

-----

**Conclusion:**
If you're using Airflow 2.0 or later, abandon the `with DAG` approach and adopt the `@dag` decorator along with the TaskFlow API. It'll save you time, reduce code, and make your DAGs more readable and easier to maintain.


## How Airflow enables scheduling, monitoring, and retrying tasks.

Airflow, the underlying engine of Cloud Composer, is a powerful platform that excels at enabling the scheduling, monitoring, and retrying of tasks within complex workflows. It achieves this through its core components and well-defined concepts.

Here's a breakdown of how Airflow handles each of these aspects:

### Scheduling Tasks

Airflow's primary function is to schedule and execute tasks. It does this through a combination of its **Scheduler** component and the `schedule_interval` parameter defined in your DAGs.

* **`schedule_interval`:** This is the most direct way to tell Airflow when to run your DAG. It can be:
    * **CRON Expressions:** The most common and flexible way to define schedules (e.g., `'0 0 * * *'` for daily at midnight, `'@daily'`, `'@hourly'`). This allows for highly precise control over execution times.
    * **`datetime.timedelta` Objects:** For simple, fixed intervals (e.g., `timedelta(days=1)` for daily).
    * **`None`:** The DAG will only be triggered manually or externally (e.g., via the Airflow UI, REST API, or an external system).
    * **`@once`:** The DAG runs only once after `start_date`.

* **Scheduler Component:**
    * The Airflow Scheduler is a persistent process that continuously monitors DAGs and their `schedule_interval`.
    * It looks for DAGs that are due to run based on their schedule and `start_date`.
    * When a DAG run is due, the scheduler creates a **DAG Run** instance. A DAG Run represents a single execution of a DAG for a specific logical date (or data interval).
    * For each task in that DAG Run, the scheduler creates **Task Instances**. A Task Instance is a specific execution of a task for a specific DAG Run.
    * The scheduler then identifies which Task Instances are ready to run (i.e., their dependencies are met) and submits them to the **Executor**.

* **`start_date` and `end_date`:**
    * **`start_date`:** Crucial for scheduling. Airflow will only start scheduling DAG runs *after* this date. The first run will typically be for the period *after* the `start_date`.
    * **`end_date`:** (Optional) Defines when the DAG should stop being scheduled.

* **Logical Date (Execution Date):** Airflow's scheduling is based on the concept of a "logical date" or "execution date." This is the start of the data interval for which the DAG run is processing data. For example, a DAG scheduled to run daily at midnight (for the *previous* day's data) with a `start_date` of 2023-01-01 will have its first execution date for 2023-01-01, but the actual run will start shortly after 2023-01-02 00:00:00. This design ensures that all data for a specific period is available before processing begins.


In [ ]:
# Example Airflow / GCP code snippet
from __future__ import annotations

import pendulum

from airflow.decorators import dag # Import the dag decorator
from airflow.operators.bash import BashOperator

# Basic daily schedule
@dag(
    dag_id="scheduling_daily_example_decorator", # Changed dag_id to avoid conflict
    👉 start_date=pendulum.datetime(2023, 1, 1, tz="UTC"),
    👉 schedule="0 0 * * *",  # Daily at midnight UTC
    catchup=False,  # Don't run for past missed schedules
    tags=["scheduling", "decorator"],
)
def daily_scheduled_dag():
    task_daily = BashOperator(
        task_id="print_daily_execution_date",
        bash_command='echo "Daily DAG Run for: {{ ds }}"',
    )
    # No need to assign tasks to 'dag' object explicitly

# Instantiate the DAG by calling the decorated function
daily_scheduled_dag()


# Hourly schedule
@dag(
    dag_id="scheduling_hourly_example_decorator", # Changed dag_id
    start_date=pendulum.datetime(2023, 1, 1, tz="UTC"),
    👉 schedule="0 * * * *",  # Every hour at minute 0
    catchup=False,
    tags=["scheduling", "decorator"],
)
def hourly_scheduled_dag():
    task_hourly = BashOperator(
        task_id="print_hourly_execution_date",
        bash_command='echo "Hourly DAG Run for: {{ ds }} {{ ts }}"',
    )

hourly_scheduled_dag()


# Schedule using timedelta (for simpler intervals)
@dag(
    dag_id="scheduling_timedelta_example_decorator", # Changed dag_id
    start_date=pendulum.datetime(2023, 1, 1, tz="UTC"),
    👉 schedule=pendulum.duration(hours=2), # Every 2 hours
    catchup=False,
    tags=["scheduling", "decorator"],
)
def timedelta_scheduled_dag():
    task_timedelta = BashOperator(
        task_id="print_timedelta_execution_date",
        bash_command='echo "Timedelta DAG Run for: {{ ds }} {{ ts }}"',
    )

timedelta_scheduled_dag()


# Manual-only DAG (no automatic schedule)
@dag(
    dag_id="scheduling_manual_only_example_decorator", # Changed dag_id
    👉 start_date=pendulum.datetime(2023, 1, 1, tz="UTC"),
    schedule=None,  # No automatic schedule
    catchup=False,
    tags=["scheduling", "decorator"],
)
def manual_only_dag():
    task_manual = BashOperator(
        task_id="print_manual_trigger_info",
        bash_command='echo "Manual DAG Triggered. Run ID: {{ run_id }}"',
    )

manual_only_dag()


# One-time run DAG (runs once after start_date, then never again)
@dag(
    dag_id="scheduling_one_time_example_decorator", # Changed dag_id
    start_date=pendulum.datetime(2023, 1, 1, tz="UTC"),
    👉 schedule="@once", # Runs only once
    catchup=False,
    tags=["scheduling", "decorator"],
)
def one_time_dag():
    task_once = BashOperator(
        task_id="print_one_time_info",
        bash_command='echo "This DAG runs only once. Execution date: {{ ds }}"',
    )

one_time_dag()




### Monitoring Tasks

Airflow provides robust mechanisms for monitoring the status and progress of your workflows:

* **Airflow UI (Web Server):** This is the primary interface for monitoring.
    * **DAGs View:** Shows a list of all your DAGs, their last run status, schedule, and tags. ![image-2.png](../artifacts/01_airflow_ui_graph_view.png)
    * **Graph View:** A visual representation of your DAG, showing all tasks and their dependencies. Each task node is color-coded to indicate its current status (running, success, failed, skipped, upstream_failed, etc.). ![image.png](../artifacts/01_airflow_ui_grid_view.png)
    * **Tree View:** Provides a compact overview of multiple DAG runs and the status of tasks within them.
    * **Gantt Chart:** Visualizes the duration of tasks across DAG runs, helping identify bottlenecks.![image-3.png](../artifacts/01_airflow_ui_gantt_view.png)
    * **Logs:** For every Task Instance execution, Airflow captures stdout/stderr. You can easily view these logs directly in the Airflow UI, which is invaluable for debugging.
    * **XComs (Cross-communication):** You can inspect values pushed between tasks using XComs in the UI, which helps trace data flow.
    * **Task Instance Details:** Click on any task instance in the UI to see its execution details, run history, and parameters.

* **Metrics (via Prometheus, StatsD, Datadog etc.):** Airflow exposes internal metrics (e.g., task success/failure rates, scheduler latency, executor queue size). These can be scraped by monitoring systems like Prometheus and visualized in dashboards (e.g., Grafana) to provide comprehensive operational insights. Cloud Composer integrates these with Cloud Monitoring.

* **External Logging:** Airflow is configured to send logs to a persistent storage backend (e.g., Google Cloud Logging for Cloud Composer). This allows for centralized log analysis, searching, and alerting.

* **Alerting and Notifications:**
    * **`on_failure_callback` / `on_success_callback`:** You can define Python functions that execute when a task succeeds or fails. These callbacks can send notifications (e.g., via Slack, Email, PagerDuty, Pub/Sub) or trigger other actions.
    * **`email_on_failure` / `email_on_retry`:** Parameters in `default_args` or task definitions to automatically send email notifications on failure or retry.
    * **Sensors:** While not directly for monitoring *tasks*, sensors can monitor external conditions (e.g., file arrival in GCS, BigQuery table existence) and trigger subsequent tasks only when those conditions are met.



### Retrying Tasks

Airflow has robust built-in mechanisms for automatically retrying failed tasks, making your pipelines resilient to transient failures.

* **`retries` Parameter:**
    * Defined in `default_args` (applied to all tasks in a DAG) or on individual task instances.
    * Specifies the maximum number of times a task should be retried before being marked as failed.
    * **Example:** `retries=3` means the task will attempt to run up to 4 times (1 initial attempt + 3 retries) before failing.

* **`retry_delay` Parameter:**
    * Also defined in `default_args` or on individual tasks.
    * Specifies the time to wait between retry attempts. This is often a `datetime.timedelta` object.
    * **Example:** `retry_delay=timedelta(minutes=5)` will wait 5 minutes before the next retry.

* **`retry_exponential_backoff`:**
    * A boolean parameter. If set to `True`, the `retry_delay` will increase exponentially for each subsequent retry (e.g., 1 min, 2 min, 4 min, 8 min). This is useful for transient issues that might resolve over time.

* **`max_active_runs` (at DAG level):** While not directly a retry mechanism, this limits the number of concurrent DAG runs. If a DAG fails and is configured to retry, and `max_active_runs` is low, it might queue up behind other active runs or previous failed runs, indirectly affecting when retries can occur.

* **How Retries Work:**
    1.  A task instance fails its initial execution.
    2.  If `retries` > 0, the task instance's state changes to `up_for_retry`.
    3.  The scheduler waits for `retry_delay` to elapse.
    4.  After the delay, the scheduler re-queues the task instance for execution.
    5.  This process repeats until the task succeeds or the `retries` limit is reached, at which point the task instance is marked as `failed`.

* **Idempotency:** When designing tasks that will be retried, it's crucial to make them **idempotent**. This means that running the task multiple times with the same inputs should produce the exact same outcome without unintended side effects. For example, if a task creates a file, ensure it either creates it safely (e.g., with unique names) or handles existing files gracefully if re-executed.



In [ ]:
# Example Airflow / GCP code snippet
from __future__ import annotations

import random
import pendulum

from airflow.decorators import dag # Import the dag decorator
from airflow.operators.python import PythonOperator
from airflow.exceptions import AirflowException

def _unreliable_task_logic():
    """
    A Python function that simulates an unreliable process.
    It will fail randomly, but eventually succeed after a few retries.
    """
    from airflow.operators.python import get_current_context
    context = get_current_context()
    current_attempt = context['ti'].try_number # try_number starts at 1 for initial run

    # Simulate success after 3 attempts (1 initial + 2 retries)
    if current_attempt >= 3:
        print(f"Attempt {current_attempt}: Finally, success! (This is the 3rd attempt or later)")
        return "Task succeeded!"
    else:
        # Simulate a random failure for demonstration
        if random.random() < 0.7:  # 70% chance to fail on initial attempts
            print(f"Attempt {current_attempt}: Oh no! Simulated failure.")
            raise AirflowException("Simulated transient error")
        else:
            print(f"Attempt {current_attempt}: Lucky shot! Succeeded unexpectedly.")
            return "Task succeeded unexpectedly early!"


@dag(
    dag_id="retrying_example_decorator", # Changed dag_id
    start_date=pendulum.datetime(2023, 1, 1, tz="UTC"),
    schedule=None,  # Manual trigger for demonstration
    catchup=False,
    tags=["retries", "decorator"],
    default_args={
        'owner': 'airflow',
        👉 'retries': 5, # Allow up to 5 retries (total 6 attempts)
        👉 'retry_delay': pendulum.duration(minutes=1), # Wait 1 minute between retries
        👉 'retry_exponential_backoff': True, # Increase retry delay exponentially
        👉 'max_retry_delay': pendulum.duration(minutes=10) # Cap the exponential backoff
    }
)
def retrying_pipeline_dag():
    unreliable_task = PythonOperator(
        task_id="my_unreliable_task",
        python_callable=_unreliable_task_logic,
    )

# Instantiate the DAG
retrying_pipeline_dag()


## Environment variables 

They play a crucial role in Apache Airflow for several reasons: configuration, flexibility, and managing sensitive information. Understanding how to use them effectively is key to building robust and maintainable data pipelines.

### What are Environment Variables in Airflow?

At its core, an environment variable is a dynamic-named value that can affect the way running processes behave on a computer. In the context of Airflow, these variables are typically set in the operating system environment where your Airflow components (scheduler, webserver, workers, triggerer) are running.

### Key Use Cases for Environment Variables in Airflow

1.  **Airflow Configuration Overrides:**
    Airflow's behavior is primarily controlled by its `airflow.cfg` file. However, for dynamic deployments (like Docker, Kubernetes, or managed services like Composer), directly modifying `airflow.cfg` isn't always practical or desirable. Environment variables provide a way to override these settings.

    **Convention:** `AIRFLOW__<SECTION>__<KEY>`

      * The `AIRFLOW__` prefix indicates an Airflow configuration.
      * `<SECTION>` corresponds to a section header in `airflow.cfg` (e.g., `core`, `database`, `webserver`).
      * `<KEY>` corresponds to a configuration option within that section.
      * Double underscores (`__`) are used to separate the prefix, section, and key.

    **Example:**
    To set the `executor` in the `core` section of `airflow.cfg` to `KubernetesExecutor`, you would set the environment variable.This is especially useful for setting things like:

      * Database connection strings (`AIRFLOW__DATABASE__SQL_ALCHEMY_CONN`)
      * Executor type (`AIRFLOW__CORE__EXECUTOR`)
      * Scheduler interval (`AIRFLOW__SCHEDULER__DAG_DIR_LIST_INTERVAL`)
      * Logging configurations (`AIRFLOW__CORE__LOGGING_LEVEL`)


In [ ]:
# Override Airflow core configuration using the AIRFLOW__<SECTION>__<KEY> convention
export AIRFLOW__CORE__EXECUTOR=KubernetesExecutor


2.  **Passing Runtime Parameters to DAGs/Tasks:**
    You might have parameters that your DAGs or individual tasks need at runtime, but which shouldn't be hardcoded into the Python file.

    - **Accessing in Python:** Inside your DAGs or Python callables, you can access these variables using Python's `os` module:


In [ ]:
# Read an environment variable inside a Python task with a safe fallback value

    import os

    def my_python_task_callable():
        my_custom_value = os.getenv("MY_CUSTOM_VAR", "default_value_if_not_set")
        print(f"My custom value is: {my_custom_value}")

    # In your DAG:
    my_task = PythonOperator(
        task_id='process_data',
        python_callable=my_python_task_callable,
    )


**Passing to BashOperator (via `env` parameter):**
    The `BashOperator` (and other operators that accept an `env` parameter) allows you to explicitly pass environment variables that will be available *only* to the subprocess executed by that specific task instance.


In [ ]:
# Read an environment variable inside a Python task with a safe fallback value
my_bash_task = BashOperator(
    task_id='run_script',
    bash_command='echo "My secret is $MY_TASK_SECRET"',
    env={"MY_TASK_SECRET": os.getenv("GLOBAL_SECRET_FROM_ENV")} # Pass a global env var
)


Or, you can directly set an environment variable within the `env` dictionary:


In [ ]:
# Define a task-scoped environment variable directly inside the BashOperator env dict
my_bash_task_direct = BashOperator(
    task_id='run_script_direct',
    bash_command='echo "Task-specific variable is $TASK_VAR"',
    env={"TASK_VAR": "a_value_for_this_task_only"}
)



3.  **Managing Sensitive Information (with caveats):**
    While Airflow has dedicated "Variables" and "Connections" (stored in the metadata database, often encrypted with Fernet, or via Secrets Backends), environment variables can also be used for sensitive data, especially in containerized environments where they might be injected as Kubernetes Secrets or Docker Compose environment variables.

    **Airflow's convention for Variables from ENV:**
    Airflow (since 1.10.10) allows you to set Airflow "Variables" using environment variables with the prefix `AIRFLOW_VAR_`.


In [ ]:
# Define Airflow Variables via environment variables using the AIRFLOW_VAR_<KEY> prefix
export AIRFLOW_VAR_MY_API_KEY=your_actual_api_key
export AIRFLOW_VAR_DATABASE_CONFIG='{"host": "db.example.com", "port": 5432}'



These will then be accessible within your DAGs using `Variable.get()` or Jinja templating, just like regular Airflow Variables, but they **will take precedence** over any Variable set in the Airflow UI/CLI.


In [ ]:
# Retrieve an Airflow Variable inside a Python task at runtime
from airflow.models import Variable

def use_api_key_task():
    api_key = Variable.get("MY_API_KEY")
    db_config_json = Variable.get("DATABASE_CONFIG", deserialize_json=True)
    print(f"API Key: {api_key}, DB Host: {db_config_json['host']}")

# Or in BashOperator with Jinja templating for late rendering:
bash_task_templated = BashOperator(
    task_id="templated_env_var",
    bash_command='echo "The API Key is {{ var.value.MY_API_KEY }}"'
)


### Best Practices and Considerations

  * **Security:**

      * **Visibility:** Environment variables set directly on the system or in Dockerfiles/Kubernetes deployments are **not masked or hidden in the Airflow UI or logs by default** when accessed via `os.getenv()`. This is a significant security risk for sensitive data.
      * **`AIRFLOW_VAR_` prefix masking:** When using the `AIRFLOW_VAR_` prefix to set Airflow Variables via environment variables, these values are *not* displayed in the Airflow UI's "Variables" page, and some versions of Airflow might not mask them in logs by default if retrieved directly via `os.getenv()`.
      * **Secrets Backends:** For true production-grade secret management, Airflow strongly recommends using a **Secrets Backend** (e.g., HashiCorp Vault, AWS Secrets Manager, Google Secret Manager, Azure Key Vault). These integrate with Airflow to fetch secrets dynamically at runtime, providing better security practices like rotation, auditing, and fine-grained access control.
      * **KubernetesPodOperator:** Be extremely careful passing secrets as environment variables directly to `KubernetesPodOperator`'s `env` parameter, as these can easily become visible in Kubernetes pod definitions. Use Kubernetes Secrets mounted as environment variables or volumes instead.

  * **Performance:**

      * Accessing `os.getenv()` is generally faster than `Variable.get()` because `Variable.get()` makes a call to the Airflow metadata database.
      * However, avoid using `Variable.get()` in the top-level DAG code (outside of task definitions or Python callables) if it's not templated, as it can lead to performance issues due to repeated database calls every time the DAG file is parsed by the scheduler (which happens frequently). Use Jinja templating (`{{ var.value.MY_VAR }}`) in templated fields for late rendering.

  * **Deployment Consistency:** Environment variables are excellent for ensuring consistent configurations across different deployment environments (development, staging, production) without changing the DAG code itself.

  * **Immutability:** Once set, environment variables typically remain constant for the lifetime of the process or container. For values that change frequently or need to be managed through the Airflow UI, Airflow Variables (stored in the metastore) are more suitable.

